# Reasoning Scaling Laws

Training and benchmark pipeline testing H1 (data efficiency vs. model size) and H2 (agentic compute vs. model scale) on GSM8K.

**Run order:** Cell 1 (install) → Cell 2 (drive + files) → Cell 3 (GPU check) → Cell 4 (config) → Cell 5 (train 3B) → Cell 6 (train 7B) → Cell 7 (verify) → Cell 8 (benchmark) → Cell 9 (results).

Cells 5 and 6 can be run in separate Colab sessions — checkpoints are saved to Drive after every epoch.

In [ ]:
# Cell 1 — Install dependencies (run once per session)
!pip install -q unsloth trl datasets bitsandbytes matplotlib accelerate
!pip install -q --upgrade "transformers==5.5.0"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.0/71.0 MB 17.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 30.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 38.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 16.6 MB/s eta 0:00:0000:01:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 20.4 MB/s eta 0:00:0000:0100:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 67.1 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 868.6/868.6 kB 46.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 65.7 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 69.7 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 25.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 64.8 MB/s e

In [ ]:
# Cell 2 — Mount Google Drive and copy source files to /content
import os
import sys
import shutil
from google.colab import drive

drive.mount('/content/drive')

DRIVE_PROJECT = '/content/drive/MyDrive/ExamensArbete'
CONTENT_DIR   = '/content'

SOURCE_FILES = [
    'utils.py',
    'prompts.py',
    'rewards.py',
    'grpo_trainer.py',
    'agent.py',
    'benchmarker.py',
    'warmup_data.json',
]

print('Copying source files from Drive to /content ...')
for fname in SOURCE_FILES:
    src = os.path.join(DRIVE_PROJECT, fname)
    dst = os.path.join(CONTENT_DIR, fname)
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f'  OK  {fname}')
    else:
        print(f'  MISSING  {fname} -- upload to {DRIVE_PROJECT} and re-run this cell')

os.chdir(CONTENT_DIR)
if CONTENT_DIR not in sys.path:
    sys.path.insert(0, CONTENT_DIR)

print(f'Working directory: {os.getcwd()}')

In [ ]:
# Cell 3 — GPU diagnostics
import torch

if torch.cuda.is_available():
    name  = torch.cuda.get_device_name(0)
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    free  = (torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated(0)) / 1e9
    print(f'GPU   : {name}')
    print(f'VRAM  : {total:.1f} GB total  |  {free:.1f} GB free')
    print(f'BF16  : {torch.cuda.is_bf16_supported()}')
else:
    print('WARNING: No GPU detected. Switch runtime to L4 GPU in Runtime > Change runtime type.')

In [ ]:
# Cell 4 — Configuration
MODEL_3B        = 'unsloth/Qwen2.5-3B-Instruct-bnb-4bit'
MODEL_7B        = 'unsloth/Qwen2.5-7B-Instruct-bnb-4bit'
DATA_FRACTIONS  = [0.10, 0.20, 0.40]
WARMUP_PATH     = 'warmup_data.json'
SEED            = 42
MAX_EPOCHS      = 3

DRIVE_SAVE_DIR  = '/content/drive/MyDrive/ExamensArbete/checkpoints'
BENCHMARK_DIR   = '/content/drive/MyDrive/ExamensArbete/benchmark_results'

print('Config:')
print(f'  3B model       : {MODEL_3B}')
print(f'  7B model       : {MODEL_7B}')
print(f'  Data fractions : {[int(f*100) for f in DATA_FRACTIONS]}%')
print(f'  Checkpoint dir : {DRIVE_SAVE_DIR}')

## Test Run (optional)

Run the cell below **instead of Cells 5–8** to do a fast end-to-end check of the whole pipeline.
It trains the 3B model on 1% of the data (≈74 examples, 1 epoch) then runs inference on 20 questions.
Total time on an L4 should be under 15 minutes.

If this cell completes without errors the full pipeline is safe to run.

In [ ]:
# Test Cell — end-to-end pipeline check
# Trains 3B on 1% data then runs a 20-question mini benchmark.
# Run Cells 1-4 first so that dependencies are installed and config variables are set.

import gc
import os
import torch
from grpo_trainer import run_grpo
from benchmarker import (
    load_benchmark_questions,
    _load_trained_model,
    _run_nonagentic,
    _format_trained_prompt,
    _compute_metrics,
)
from utils import extract_tagged_answer

TEST_FRACTION    = 0.01   # ~74 training examples
TEST_CHECKPOINT  = f'{DRIVE_SAVE_DIR}/grpo_3b_1pct_best'
TEST_N_QUESTIONS = 20

# ── Step 1: Train ──────────────────────────────────────────────────────────────
print('=' * 60)
print('STEP 1: Training 3B on 1% data (1 epoch max)')
print('=' * 60)

if os.path.exists(TEST_CHECKPOINT):
    print(f'Checkpoint already exists at {TEST_CHECKPOINT} — skipping training.')
else:
    model, tokenizer, best_acc = run_grpo(
        model_name=MODEL_3B,
        data_fraction=TEST_FRACTION,
        save_dir=DRIVE_SAVE_DIR,
        warmup_path=WARMUP_PATH,
        max_epochs=1,   # single epoch keeps the test fast
        seed=SEED,
    )
    del model, tokenizer
    gc.collect()
    torch.cuda.empty_cache()
    print(f'\nTraining complete. Best val accuracy: {best_acc:.4f}')

# ── Step 2: Mini benchmark (non-agentic, 20 questions) ─────────────────────────
print()
print('=' * 60)
print(f'STEP 2: Mini benchmark ({TEST_N_QUESTIONS} questions, 3B no agent)')
print('=' * 60)

questions, solutions = load_benchmark_questions(TEST_N_QUESTIONS, seed=SEED)
model, tokenizer = _load_trained_model(TEST_CHECKPOINT)
results = _run_nonagentic(
    model, tokenizer, questions, solutions,
    _format_trained_prompt, extract_tagged_answer,
)
del model, tokenizer
gc.collect()
torch.cuda.empty_cache()

metrics = _compute_metrics(results)
print(f'\nTest results ({TEST_N_QUESTIONS} questions):')
print(f'  Accuracy   : {metrics["accuracy"]:.2%}  ({metrics["correct"]}/{metrics["total"]})')
print(f'  Avg tokens : {metrics["avg_tokens"]:.0f}')
print()
print('✓ Pipeline completed without errors.')
print('  Note: accuracy will be low at 1% data — that is expected.')
print('  A non-zero accuracy (> 0) means the reward signal is working.')

## Training

Cells 5 and 6 each run 3 independent GRPO training runs (one per data fraction).
Each run: SFT warmup (1 epoch) → GRPO with early stopping (max 3 epochs).
Checkpoints are saved to Drive after every epoch — safe to resume if the session disconnects.
If a `_best` checkpoint already exists for a run, that run is skipped automatically.

In [ ]:
# Cell 5 — Train 3B model across all data fractions
import gc
import os
import torch
from grpo_trainer import run_grpo

for fraction in DATA_FRACTIONS:
    pct      = int(fraction * 100)
    n_ex     = int(7500 * fraction)
    expected = f'{DRIVE_SAVE_DIR}/grpo_3b_{pct}pct_best'

    print(f'\n{"="*60}')
    print(f'3B | {pct}% data  ({n_ex} examples)')
    print(f'{"="*60}')

    if os.path.exists(expected):
        print(f'  Checkpoint already exists at {expected} -- skipping.')
        continue

    model, tokenizer, best_acc = run_grpo(
        model_name=MODEL_3B,
        data_fraction=fraction,
        save_dir=DRIVE_SAVE_DIR,
        warmup_path=WARMUP_PATH,
        max_epochs=MAX_EPOCHS,
        seed=SEED,
    )
    del model, tokenizer
    gc.collect()
    torch.cuda.empty_cache()
    print(f'  Completed. Best val accuracy: {best_acc:.4f}')

In [ ]:
# Cell 6 — Train 7B model across all data fractions
import gc
import os
import torch
from grpo_trainer import run_grpo

for fraction in DATA_FRACTIONS:
    pct      = int(fraction * 100)
    n_ex     = int(7500 * fraction)
    expected = f'{DRIVE_SAVE_DIR}/grpo_7b_{pct}pct_best'

    print(f'\n{"="*60}')
    print(f'7B | {pct}% data  ({n_ex} examples)')
    print(f'{"="*60}')

    if os.path.exists(expected):
        print(f'  Checkpoint already exists at {expected} -- skipping.')
        continue

    model, tokenizer, best_acc = run_grpo(
        model_name=MODEL_7B,
        data_fraction=fraction,
        save_dir=DRIVE_SAVE_DIR,
        warmup_path=WARMUP_PATH,
        max_epochs=MAX_EPOCHS,
        seed=SEED,
    )
    del model, tokenizer
    gc.collect()
    torch.cuda.empty_cache()
    print(f'  Completed. Best val accuracy: {best_acc:.4f}')

## Benchmark

Uses the 40% checkpoint for Groups 1-4. Group 5 is the zero-shot Qwen2.5-14B-Instruct baseline.
500 questions from the official GSM8K test split (fixed seed = 42).

In [ ]:
# Cell 7 — Verify all required checkpoints exist
import os

to_check = {
    '3B 10%': f'{DRIVE_SAVE_DIR}/grpo_3b_10pct_best',
    '3B 20%': f'{DRIVE_SAVE_DIR}/grpo_3b_20pct_best',
    '3B 40%': f'{DRIVE_SAVE_DIR}/grpo_3b_40pct_best',
    '7B 10%': f'{DRIVE_SAVE_DIR}/grpo_7b_10pct_best',
    '7B 20%': f'{DRIVE_SAVE_DIR}/grpo_7b_20pct_best',
    '7B 40%': f'{DRIVE_SAVE_DIR}/grpo_7b_40pct_best',
}

all_ok = True
for label, path in to_check.items():
    status = 'OK     ' if os.path.exists(path) else 'MISSING'
    print(f'  [{status}]  {label}  ->  {path}')
    if 'MISSING' in status:
        all_ok = False

print()
if all_ok:
    print('All checkpoints present. Ready to run benchmark.')
else:
    print('Some checkpoints are missing. Complete training before running the benchmark.')

In [ ]:
# Cell 8 — Run benchmark (all 5 groups, 500 questions)
from benchmarker import run_benchmark

CHECKPOINT_3B = f'{DRIVE_SAVE_DIR}/grpo_3b_40pct_best'
CHECKPOINT_7B = f'{DRIVE_SAVE_DIR}/grpo_7b_40pct_best'

summaries = run_benchmark(
    checkpoint_3b=CHECKPOINT_3B,
    checkpoint_7b=CHECKPOINT_7B,
    save_dir=BENCHMARK_DIR,
    n_questions=500,
    seed=SEED,
)

In [ ]:
# Cell 9 — Display results table and plot
import pandas as pd
from IPython.display import Image, display

df = pd.read_csv(f'{BENCHMARK_DIR}/benchmark_summary.csv')
df['accuracy_%']         = (df['accuracy'] * 100).round(2)
df['avg_tokens']         = df['avg_tokens'].round(1)
df['tokens_per_correct'] = df['tokens_per_correct'].round(1)

display(df[['group_name', 'accuracy_%', 'avg_tokens', 'tokens_per_correct', 'correct', 'total']])

print()
display(Image(filename=f'{BENCHMARK_DIR}/benchmark_plot.png'))